# Лабораторная работа №2

## Формирование отчётов в Apache Spark

Notebook подготовлен для запуска в Google Colab. Он строит отчёт по 10 наиболее популярным языкам программирования для каждого года и сохраняет результат в формате Apache Parquet.

Для быстрого запуска достаточно загрузить в Colab файлы `posts_sample.xml` и `programming-languages.csv`. Для полного отчёта по годам 2010-2020 лучше загрузить полный `Posts.xml` из архива Stack Overflow.


In [1]:
!pip -q install pyspark



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import csv
import re
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    files = None
from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (
    SparkSession.builder
    .appName("Lab2_Colab_Report")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext

print("Spark version:", spark.version)


Spark version: 4.0.2


### Загрузка входных файлов

После запуска ячейки загрузите:
- `posts_sample.xml` или полный `Posts.xml`
- `programming-languages.csv`


In [3]:
if files is not None:
    uploaded = files.upload()
else:
    uploaded = {}

available_files = sorted(Path('.').glob('*'))
print([path.name for path in available_files if path.is_file()][:20])


['example_of_runing_report_script_in_azure.png', 'hints.md', 'lab_2_colab.ipynb', 'README.md', 'top_languages_report_parquet.zip']


In [4]:
candidate_posts = ["Posts.xml", "posts.xml", "posts_sample.xml", "../data/posts_sample.xml"]
candidate_languages = ["programming-languages.csv", "../data/programming-languages.csv"]

posts_path = next((name for name in candidate_posts if Path(name).exists()), None)
languages_path = next((name for name in candidate_languages if Path(name).exists()), None)

if posts_path is None:
    raise FileNotFoundError("Не найден файл Posts.xml или posts_sample.xml")
if languages_path is None:
    raise FileNotFoundError("Не найден файл programming-languages.csv")

print("posts_path =", posts_path)
print("languages_path =", languages_path)


posts_path = ../data/posts_sample.xml
languages_path = ../data/programming-languages.csv


In [5]:
def canonical_display_name(name: str) -> str:
    value = name.strip().replace("–", "-")
    value = re.sub(r"\s*\(.*?\)", "", value)
    value = re.sub(r"\s*-\s*.*$", "", value)
    return value.strip()


def normalized_token(value: str) -> str:
    value = value.strip().lower().replace("–", "-")
    value = re.sub(r"\s*\(.*?\)", "", value)
    value = re.sub(r"\s*-\s*.*$", "", value)
    value = value.replace("'", "")
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def build_aliases(raw_name: str):
    canonical = canonical_display_name(raw_name)
    base = normalized_token(canonical)
    aliases = {
        base,
        base.replace(" ", "-"),
        base.replace(" ", ""),
        base.replace(" ", "."),
    }

    extra_aliases = {
        "c#": {"c#", "c-sharp"},
        "c++": {"c++", "cpp"},
        "f#": {"f#", "f-sharp"},
        "go": {"go", "golang"},
        "objective-c": {"objective-c", "objectivec"},
        "visual basic .net": {"vb.net", "visual-basic-.net", "visual-basic"},
        "assembly language": {"assembly", "assembly-language"},
        "common lisp": {"common-lisp", "lisp"},
        "emacs lisp": {"emacs-lisp", "elisp"},
        "wolfram language": {"mathematica", "wolfram-language"},
    }
    aliases.update(extra_aliases.get(base, set()))
    return canonical, {alias for alias in aliases if alias}


alias_to_language = {}
languages = []
with open(languages_path, encoding="utf-8") as csv_file:
    reader = csv.DictReader(csv_file)
    for row in reader:
        canonical, aliases = build_aliases(row["name"])
        languages.append(canonical)
        for alias in aliases:
            alias_to_language.setdefault(alias, canonical)

print("Languages loaded:", len(languages))
print("Aliases loaded:", len(alias_to_language))
print("Examples:", {key: alias_to_language[key] for key in ['python', 'java', 'javascript', 'c#', 'c++', 'go'] if key in alias_to_language})


Languages loaded: 700
Aliases loaded: 957
Examples: {'python': 'Python', 'java': 'Java', 'javascript': 'JavaScript', 'c#': 'C#', 'c++': 'C++', 'go': 'Go'}


In [6]:
def is_post_row(line: str) -> bool:
    stripped = line.strip().lstrip("\ufeff")
    return stripped.startswith("<row ")


def parse_post_row(line: str):
    try:
        row = ET.fromstring(line.strip().lstrip("\ufeff"))
        attrs = row.attrib
        return (
            attrs.get("Id"),
            attrs.get("PostTypeId"),
            attrs.get("CreationDate"),
            attrs.get("Tags")
        )
    except Exception:
        return None


schema = T.StructType([
    T.StructField("post_id", T.StringType(), True),
    T.StructField("post_type_id", T.StringType(), True),
    T.StructField("creation_date", T.StringType(), True),
    T.StructField("tags_raw", T.StringType(), True),
])

posts_rdd = (
    sc.textFile(posts_path)
    .filter(is_post_row)
    .map(parse_post_row)
    .filter(lambda row: row is not None)
)

posts_df = spark.createDataFrame(posts_rdd, schema=schema)
posts_df.show(5, truncate=False)
print("Rows parsed:", posts_df.count())


+-------+------------+-----------------------+------------------------------------------------------+
|post_id|post_type_id|creation_date          |tags_raw                                              |
+-------+------------+-----------------------+------------------------------------------------------+
|4      |1           |2008-07-31T21:42:52.667|<c#><floating-point><type-conversion><double><decimal>|
|6      |1           |2008-07-31T22:08:08.620|<html><css><internet-explorer-7>                      |
|7      |2           |2008-07-31T22:17:57.883|NULL                                                  |
|9      |1           |2008-07-31T23:40:59.743|<c#><.net><datetime>                                  |
|11     |1           |2008-07-31T23:55:37.967|<c#><datetime><time><datediff><relative-time-span>    |
+-------+------------+-----------------------+------------------------------------------------------+
only showing top 5 rows


Rows parsed: 46006


In [7]:
questions_df = (
    posts_df
    .filter(F.col("post_type_id") == "1")
    .filter(F.col("tags_raw").isNotNull())
    .withColumn("creation_ts", F.to_timestamp("creation_date", "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .withColumn("year", F.year("creation_ts"))
    .filter((F.col("year") >= 2010) & (F.col("year") <= 2020))
    .select("post_id", "year", "tags_raw")
)

questions_df.show(5, truncate=False)
print("Question rows in 2010-2020:", questions_df.count())


+-------+----+--------------------------------------+
|post_id|year|tags_raw                              |
+-------+----+--------------------------------------+
|3768363|2010|<c++><character-encoding>             |
|3775996|2010|<sharepoint><infopath>                |
|3776721|2010|<iphone><app-store><in-app-purchase>  |
|3777993|2010|<symfony1><schema><doctrine><fixtures>|
|3778222|2010|<java>                                |
+-------+----+--------------------------------------+
only showing top 5 rows


Question rows in 2010-2020: 17642


In [8]:
aliases_df = spark.createDataFrame(
    [(tag, language) for tag, language in alias_to_language.items()],
    ["tag", "language"]
)

language_mentions_df = (
    questions_df
    .withColumn("tag", F.explode(F.expr("regexp_extract_all(tags_raw, '<([^<>]+)>', 1)")))
    .withColumn("tag", F.lower(F.trim(F.col("tag"))))
    .join(F.broadcast(aliases_df), on="tag", how="inner")
    .select("post_id", "year", "language")
    .distinct()
    .select("year", "language")
)

print("Language mentions:", language_mentions_df.count())
language_mentions_df.show(10, truncate=False)


Language mentions: 10071


+----+----------+
|year|language  |
+----+----------+
|2011|PHP       |
|2016|PHP       |
|2018|Swift     |
|2011|Java      |
|2017|Python    |
|2012|C#        |
|2012|JavaScript|
|2018|C#        |
|2010|C         |
|2014|PHP       |
+----+----------+
only showing top 10 rows


In [9]:
language_stats_df = (
    language_mentions_df
    .groupBy("year", "language")
    .agg(F.count(F.lit(1)).alias("posts_count"))
)

window = Window.partitionBy("year").orderBy(F.desc("posts_count"), F.asc("language"))

top10_by_year_df = (
    language_stats_df
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") <= 10)
    .select("year", "rank", "language", "posts_count")
    .orderBy("year", "rank")
)

top10_by_year_df.show(200, truncate=False)


+----+----+-----------+-----------+
|year|rank|language   |posts_count|
+----+----+-----------+-----------+
|2010|1   |C#         |96         |
|2010|2   |Java       |52         |
|2010|3   |PHP        |46         |
|2010|4   |JavaScript |44         |
|2010|5   |C++        |28         |
|2010|6   |Python     |26         |
|2010|7   |C          |20         |
|2010|8   |Ruby       |12         |
|2010|9   |Delphi     |8          |
|2010|10  |AppleScript|3          |
|2011|1   |PHP        |102        |
|2011|2   |C#         |100        |
|2011|3   |Java       |93         |
|2011|4   |JavaScript |83         |
|2011|5   |C++        |42         |
|2011|6   |Python     |37         |
|2011|7   |C          |24         |
|2011|8   |Ruby       |20         |
|2011|9   |Perl       |9          |
|2011|10  |Delphi     |8          |
|2012|1   |PHP        |154        |
|2012|2   |C#         |138        |
|2012|3   |JavaScript |132        |
|2012|4   |Java       |124        |
|2012|5   |C++        |81   

In [10]:
output_path = "top_languages_report_parquet"

top10_by_year_df.write.mode("overwrite").parquet(output_path)
print("Parquet report saved to:", output_path)

report_df = spark.read.parquet(output_path).orderBy("year", "rank")
report_df.show(200, truncate=False)


Parquet report saved to: top_languages_report_parquet


+----+----+-----------+-----------+
|year|rank|language   |posts_count|
+----+----+-----------+-----------+
|2010|1   |C#         |96         |
|2010|2   |Java       |52         |
|2010|3   |PHP        |46         |
|2010|4   |JavaScript |44         |
|2010|5   |C++        |28         |
|2010|6   |Python     |26         |
|2010|7   |C          |20         |
|2010|8   |Ruby       |12         |
|2010|9   |Delphi     |8          |
|2010|10  |AppleScript|3          |
|2011|1   |PHP        |102        |
|2011|2   |C#         |100        |
|2011|3   |Java       |93         |
|2011|4   |JavaScript |83         |
|2011|5   |C++        |42         |
|2011|6   |Python     |37         |
|2011|7   |C          |24         |
|2011|8   |Ruby       |20         |
|2011|9   |Perl       |9          |
|2011|10  |Delphi     |8          |
|2012|1   |PHP        |154        |
|2012|2   |C#         |138        |
|2012|3   |JavaScript |132        |
|2012|4   |Java       |124        |
|2012|5   |C++        |81   

In [11]:
archive_path = shutil.make_archive("top_languages_report_parquet", "zip", output_path)
print("Archive created:", archive_path)
if files is not None:
    files.download(archive_path)


Archive created: C:\Users\mserg\PycharmProjects\BigDataLabs\L2 - Reports with Apache Spark\top_languages_report_parquet.zip


Структура отчёта:
- `year` — год
- `rank` — место языка в топ-10 за этот год
- `language` — название языка
- `posts_count` — число вопросов Stack Overflow, где встретился этот язык
